# AGORA z=0 Dataset Inspection

This notebook has two consolidated panels.

1. Inspect all z=0 AGORA code folders with `yt`, summarize dataset metadata, DM/EM field compatibility, particle/grid counts, and component masses, then write the result to `agora_z0_dataset_inspection.csv`.
2. Reload the CSV, display gas/grid, star, high-resolution dark matter, total dark matter counts and masses, and plot the mass differences between simulation codes.


In [ ]:
from __future__ import annotations

import csv
import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import yt
except Exception as exc:
    raise RuntimeError(f"Could not import yt in the current kernel: {exc}") from exc

ROOT = Path("/home/zhaozhang/local/AGORA_work/AGORA_Data/z0")
OUTPUT_CSV = ROOT / "agora_z0_dataset_inspection.csv"
OUTPUT_PNG = ROOT / "agora_z0_component_mass_comparison.png"

CODE_FOLDERS = {
    "ARTI": "ART-I",
    "Enzo": "ENZO",
    "AREPO": "AREPO",
    "GADGET3": "GADGET-3",
    "GEAR": "GEAR",
    "CHANGA": "CHANGA",
    "G4Cal_Pablo": "GADGET-4",
}

GADGET_MPC_UNIT_BASE = {
    "UnitLength_in_cm": 3.085678e24,
    "UnitMass_in_g": 1.989e43,
    "UnitVelocity_in_cm_per_s": 100000.0,
}

AREPO_MPC_UNIT_BASE = GADGET_MPC_UNIT_BASE

FIELD_CONFIG = {
    "grid": {
        "gas_types": ["gas", "enzo", "art"],
        "star_types": ["stars", "star", "all", "io"],
        "density_names": ["density", "Density"],
        "temperature_names": ["temperature", "Temperature"],
        "electron_names": ["El_number_density", "electron_number_density", "Electron_Number_Density", "Electron_Density", "electron_density"],
        "nh_names": ["H_number_density", "H_nuclei_density", "number_density"],
    },
    "particle": {
        "gas_types": ["PartType0", "Gas", "gas"],
        "star_types": ["PartType4", "PartType1", "Stars", "stars"],
        "density_names": ["Density", "density", "particle_density"],
        "temperature_names": ["Temperature", "temperature", "GrackleTemperature", "Temperature1"],
        "electron_names": ["HII", "HeII", "HeIII", "ElectronAbundance", "electron_abundance", "electron_number_density"],
        "nh_names": ["H_number_density", "NeutralHydrogenAbundance"],
    },
}

COMPONENT_MAP = {
    "ART-I": {"gas": [], "star": ["stars"], "dm_candidates": ["specie0", "specie1", "specie2", "specie3", "specie4", "specie5"]},
    "GADGET-3": {"gas": ["PartType0"], "star": ["PartType4"], "dm_hr": ["PartType1"], "dm_candidates": ["PartType1", "PartType5"]},
    "GADGET-4": {"gas": ["PartType0"], "star": ["PartType4"], "dm_hr": ["PartType1"], "dm_candidates": ["PartType1", "PartType5"]},
    "AREPO": {"gas": ["PartType0"], "star": ["PartType4"], "dm_hr": ["PartType1"], "dm_candidates": ["PartType1", "PartType2"]},
    "GEAR": {"gas": ["PartType0"], "star": ["PartType1", "PartType4"], "dm_hr": ["PartType2"], "dm_candidates": ["PartType2", "PartType5"]},
    "CHANGA": {"gas": ["Gas", "gas"], "star": ["Stars", "stars"], "dm_hr": ["DarkMatter", "darkmatter"], "dm_candidates": ["DarkMatter", "darkmatter"]},
}


def field_exists(ds, field):
    return field in ds.field_list or field in ds.derived_field_list


def first_field(ds, ftypes, names):
    for ft in ftypes:
        for name in names:
            f = (ft, name)
            if field_exists(ds, f):
                return f
    return None


def first_position_available(ds, ftypes):
    vector_names = ["particle_position", "Coordinates"]
    triplets = [
        ("particle_position_x", "particle_position_y", "particle_position_z"),
        ("Coordinates_x", "Coordinates_y", "Coordinates_z"),
    ]
    if first_field(ds, ftypes, vector_names) is not None:
        return True
    for ft in ftypes:
        if any(all(field_exists(ds, (ft, name)) for name in triplet) for triplet in triplets):
            return True
    return False


def first_velocity_available(ds, ftypes):
    vector_names = ["particle_velocity", "Velocities"]
    triplets = [
        ("particle_velocity_x", "particle_velocity_y", "particle_velocity_z"),
        ("velocity_x", "velocity_y", "velocity_z"),
        ("x-velocity", "y-velocity", "z-velocity"),
    ]
    if first_field(ds, ftypes, vector_names) is not None:
        return True
    for ft in ftypes:
        if any(all(field_exists(ds, (ft, name)) for name in triplet) for triplet in triplets):
            return True
    return False


def unit_base_attempts(code):
    attempts = [("none", None)]
    if code == "AREPO":
        attempts.append(("arepo_mpc", AREPO_MPC_UNIT_BASE))
    if code in {"GADGET-3", "GADGET-4"}:
        attempts.append(("gadget_mpc", GADGET_MPC_UNIT_BASE))
    return attempts


def is_changa_main_file(path):
    if not path.is_file() or not path.name.startswith("ncal-"):
        return False
    parts = path.name.split(".")
    return len(parts) == 2 and parts[-1].isdigit()


def candidate_snapshots(folder, code):
    folder = Path(folder)
    if code == "ART-I":
        return sorted(folder.glob("*.d"))
    if code == "ENZO":
        return sorted(p for p in folder.glob("RD*/RD*") if p.is_file() and p.name == p.parent.name)
    if code == "AREPO":
        return sorted(p for p in folder.glob("snap_*.hdf5") if ".hsml." not in p.name and ".kdtree" not in p.name)
    if code in {"GADGET-3", "GADGET-4"}:
        return sorted(folder.glob("snapshot_*/*.0.hdf5")) + sorted(folder.glob("snapshot_*.hdf5"))
    if code == "GEAR":
        return sorted(p for p in folder.glob("*.hdf5") if ".hsml." not in p.name and ".kdtree" not in p.name)
    if code == "CHANGA":
        preferred = sorted(p for p in folder.iterdir() if is_changa_main_file(p))
        fallback = sorted(
            p for p in folder.iterdir()
            if p.is_file() and not p.name.startswith(".")
            and not any(p.name.endswith(s) for s in [".HII", ".massform", ".Metalsdot", ".ESNRate", ".kdtree"])
            and p.name not in {"wget-log", "robots.txt.tmp"}
        )
        return preferred or fallback
    return []


def load_dataset(candidate, code):
    errors = []
    for label, unit_base in unit_base_attempts(code):
        try:
            if unit_base is None:
                return yt.load(str(candidate)), label, None
            return yt.load(str(candidate), unit_base=unit_base), label, None
        except Exception as exc:
            errors.append(f"{label}: {type(exc).__name__}: {exc}")
    return None, None, "; ".join(errors[-3:])


def quantity_size(arr):
    try:
        return int(arr.size)
    except Exception:
        try:
            return int(arr.shape[0])
        except Exception:
            return None


def sum_mass(ad, field):
    arr = ad[field]
    n = quantity_size(arr)
    try:
        mass = arr.sum().to("Msun")
        mass_value = float(mass.value)
    except Exception:
        mass_value = np.nan
    return n, mass_value


def count_mass_for_types(ad, ds, ftypes):
    mass_names = ["Masses", "Mass", "particle_mass", "cell_mass"]
    entries = []
    for ft in ftypes:
        f = first_field(ds, [ft], mass_names)
        if f is None:
            continue
        n, m = sum_mass(ad, f)
        if n is None:
            continue
        entries.append({"field": str(f), "N": int(n), "M_Msun": float(m), "mean_Msun": float(m) / n if n > 0 and np.isfinite(m) else np.nan})
    if not entries:
        return None
    return {
        "N": int(sum(e["N"] for e in entries)),
        "M_Msun": float(np.nansum([e["M_Msun"] for e in entries])),
        "mean_Msun": float(np.nansum([e["M_Msun"] for e in entries]) / max(sum(e["N"] for e in entries), 1)),
        "fields": [e["field"] for e in entries],
        "entries": entries,
    }


def inspect_components(ds, code):
    ad = ds.all_data()
    out = {
        "N_gas_elem": np.nan, "M_gas_Msun": np.nan, "gas_mean_mass_Msun": np.nan,
        "N_star": np.nan, "M_star_Msun": np.nan, "star_mean_mass_Msun": np.nan,
        "N_DM_HR": np.nan, "M_DM_HR_Msun": np.nan, "dm_hr_mean_mass_Msun": np.nan,
        "N_DM_total": np.nan, "M_DM_total_Msun": np.nan, "dm_total_mean_mass_Msun": np.nan,
        "component_notes": [],
    }

    cfg = COMPONENT_MAP.get(code, {})
    gas = count_mass_for_types(ad, ds, cfg.get("gas", [])) if cfg else None
    star = count_mass_for_types(ad, ds, cfg.get("star", [])) if cfg else None
    dm_hr = count_mass_for_types(ad, ds, cfg.get("dm_hr", [])) if cfg and cfg.get("dm_hr") else None
    dm_all = count_mass_for_types(ad, ds, cfg.get("dm_candidates", [])) if cfg else None

    if gas:
        out.update({"N_gas_elem": gas["N"], "M_gas_Msun": gas["M_Msun"], "gas_mean_mass_Msun": gas["mean_Msun"]})
        out["component_notes"].append(f"gas fields={gas['fields']}")
    if star:
        out.update({"N_star": star["N"], "M_star_Msun": star["M_Msun"], "star_mean_mass_Msun": star["mean_Msun"]})
        out["component_notes"].append(f"star fields={star['fields']}")
    if dm_all:
        out.update({"N_DM_total": dm_all["N"], "M_DM_total_Msun": dm_all["M_Msun"], "dm_total_mean_mass_Msun": dm_all["mean_Msun"]})
        if dm_hr:
            out.update({"N_DM_HR": dm_hr["N"], "M_DM_HR_Msun": dm_hr["M_Msun"], "dm_hr_mean_mass_Msun": dm_hr["mean_Msun"]})
            out["component_notes"].append(f"DM fields={dm_all['fields']}; HR DM fields={dm_hr['fields']}")
        else:
            hr = min(dm_all["entries"], key=lambda e: e["mean_Msun"] if np.isfinite(e["mean_Msun"]) else np.inf)
            out.update({"N_DM_HR": hr["N"], "M_DM_HR_Msun": hr["M_Msun"], "dm_hr_mean_mass_Msun": hr["mean_Msun"]})
            out["component_notes"].append(f"DM fields={dm_all['fields']}; HR inferred as {hr['field']} with smallest mean mass")

    # Grid gas fallback.
    if not np.isfinite(out["N_gas_elem"]):
        density_field = first_field(ds, ["gas", "enzo", "art"], ["density", "Density"])
        if density_field is not None:
            rho = ad[density_field]
            out["N_gas_elem"] = quantity_size(rho)
            mass_field = first_field(ds, ["gas"], ["cell_mass", "mass"])
            if mass_field is not None:
                n, m = sum_mass(ad, mass_field)
                out["M_gas_Msun"] = m
                out["gas_mean_mass_Msun"] = m / n if n else np.nan
            out["component_notes"].append(f"grid gas counted from {density_field}")

    # ART/Enzo particle fallback: split stars and DM by creation_time if available.
    ptype_field = first_field(ds, ["all", "io"], ["particle_type"])
    pmass_field = first_field(ds, ["all", "io"], ["particle_mass", "Mass", "Masses"])
    ctime_field = first_field(ds, ["all", "io"], ["creation_time", "particle_creation_time"])
    if ptype_field and pmass_field and ctime_field and (not np.isfinite(out["N_star"]) or not np.isfinite(out["N_DM_total"])):
        ptype = np.asarray(ad[ptype_field])
        pmass = ad[pmass_field]
        ctime = ad[ctime_field]
        dm_entries = []
        n_star = 0
        m_star = 0.0
        n_dm = 0
        m_dm = 0.0
        for t in np.unique(ptype):
            mask = ptype == t
            n = int(np.sum(mask))
            m = float(pmass[mask].sum().to("Msun").value)
            mean = m / n if n else np.nan
            try:
                ctmax = ctime[mask].max()
                is_star = bool(ctmax > 0)
            except Exception:
                is_star = False
            if is_star:
                n_star += n
                m_star += m
            else:
                n_dm += n
                m_dm += m
                dm_entries.append({"ptype": int(t), "N": n, "M_Msun": m, "mean_Msun": mean})
        if not np.isfinite(out["N_star"]):
            out.update({"N_star": n_star, "M_star_Msun": m_star, "star_mean_mass_Msun": m_star / n_star if n_star else np.nan})
        if not np.isfinite(out["N_DM_total"]):
            out.update({"N_DM_total": n_dm, "M_DM_total_Msun": m_dm, "dm_total_mean_mass_Msun": m_dm / n_dm if n_dm else np.nan})
        if dm_entries and not np.isfinite(out["N_DM_HR"]):
            hr = min(dm_entries, key=lambda e: e["mean_Msun"] if np.isfinite(e["mean_Msun"]) else np.inf)
            out.update({"N_DM_HR": hr["N"], "M_DM_HR_Msun": hr["M_Msun"], "dm_hr_mean_mass_Msun": hr["mean_Msun"]})
        out["component_notes"].append("particle fallback used particle_type + creation_time")

    out["component_notes"] = " | ".join(out["component_notes"])
    return out


def particle_counts(ds):
    counts = {}
    for ptype in getattr(ds, "particle_types_raw", []):
        for field_name in ["particle_mass", "Mass", "Masses", "Coordinates", "particle_position"]:
            f = (ptype, field_name)
            if field_exists(ds, f):
                try:
                    counts[ptype] = int(ds.all_data()[f].size)
                    break
                except Exception:
                    pass
    return counts


def inspect_one_code(code_dir, code):
    folder = ROOT / code_dir
    row = {"code_dir": code_dir, "normalized_code": code, "load_ok": False, "snapshot": "", "candidate_count": 0, "error": ""}
    try:
        candidates = candidate_snapshots(folder, code)
        row["candidate_count"] = len(candidates)
        if not candidates:
            row["error"] = "No candidate snapshot files found"
            return row
        ds = None
        errors = []
        unit_label = None
        candidate_used = None
        for candidate in candidates:
            ds, unit_label, err = load_dataset(candidate, code)
            if ds is not None:
                candidate_used = candidate
                break
            errors.append(f"{candidate}: {err}")
        if ds is None:
            row["error"] = " ; ".join(errors[-3:])
            return row

        family = "grid" if code in {"ART-I", "ENZO", "RAMSES"} else "particle"
        fcfg = FIELD_CONFIG[family]
        gas_types = fcfg["gas_types"]
        star_types = fcfg["star_types"]

        gas_density = first_field(ds, gas_types, fcfg["density_names"])
        gas_temperature = first_field(ds, gas_types, fcfg["temperature_names"])
        gas_electron = first_field(ds, gas_types, fcfg["electron_names"])
        gas_nh = first_field(ds, gas_types, fcfg["nh_names"])
        star_mass = first_field(ds, star_types, ["particle_mass", "Mass", "Masses"])

        try:
            domain_width_kpc = np.asarray(ds.domain_width.to("kpc").value).tolist()
        except Exception:
            domain_width_kpc = None

        counts = particle_counts(ds)
        component = inspect_components(ds, code)
        total_particles = int(np.nansum(list(counts.values()))) if counts else np.nan
        try:
            grid_count = int(ds.index.num_grids)
        except Exception:
            grid_count = np.nan
        try:
            domain_cell_count = int(np.prod(np.asarray(ds.domain_dimensions, dtype=int)))
        except Exception:
            domain_cell_count = np.nan

        row.update({
            "load_ok": True,
            "snapshot": str(candidate_used),
            "unit_base_used": unit_label,
            "current_redshift": getattr(ds, "current_redshift", np.nan),
            "current_time": str(getattr(ds, "current_time", "")),
            "dataset_class": f"{type(ds).__module__}.{type(ds).__name__}",
            "expected_family": family,
            "recommended_backend": "gridray" if family == "grid" else "particle",
            "gas_density_field": str(gas_density),
            "gas_temperature_field": str(gas_temperature),
            "gas_electron_field": str(gas_electron),
            "gas_nh_field": str(gas_nh),
            "gas_has_position": first_position_available(ds, gas_types),
            "gas_has_velocity": first_velocity_available(ds, gas_types),
            "star_mass_field": str(star_mass),
            "star_has_position": first_position_available(ds, star_types),
            "star_has_velocity": first_velocity_available(ds, star_types),
            "domain_width_kpc": json.dumps(domain_width_kpc),
            "length_unit": str(getattr(ds, "length_unit", "")),
            "mass_unit": str(getattr(ds, "mass_unit", "")),
            "velocity_unit": str(getattr(ds, "velocity_unit", "")),
            "particle_types": json.dumps(list(getattr(ds, "particle_types_raw", []))),
            "particle_type_counts": json.dumps(counts),
            "total_particles": total_particles,
            "grid_count": grid_count,
            "domain_cell_count": domain_cell_count,
            **component,
        })
    except Exception as exc:
        row["error"] = f"{type(exc).__name__}: {exc}"
    return row


def inspect_all(root=ROOT, output_csv=OUTPUT_CSV):
    rows = []
    for code_dir, code in CODE_FOLDERS.items():
        if not (Path(root) / code_dir).exists():
            rows.append({"code_dir": code_dir, "normalized_code": code, "load_ok": False, "error": "Folder missing"})
            continue
        print(f"Inspecting {code_dir} ({code}) ...")
        rows.append(inspect_one_code(code_dir, code))

    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = sorted({key for row in rows for key in row.keys()})
    preferred = [
        "code_dir", "normalized_code", "load_ok", "snapshot", "unit_base_used", "current_redshift", "current_time",
        "expected_family", "recommended_backend", "N_gas_elem", "M_gas_Msun", "gas_mean_mass_Msun",
        "N_star", "M_star_Msun", "star_mean_mass_Msun", "N_DM_HR", "M_DM_HR_Msun", "dm_hr_mean_mass_Msun",
        "N_DM_total", "M_DM_total_Msun", "dm_total_mean_mass_Msun", "grid_count", "domain_cell_count", "total_particles",
    ]
    fieldnames = preferred + [f for f in fieldnames if f not in preferred]
    with output_csv.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)
    print("Saved:", output_csv)
    return rows


## 1. Scan Datasets and Write the Inspection CSV

Run this cell whenever the dataset folders or the inspection logic change. It writes the merged metadata, compatibility, particle/grid count, and mass table to:

`/Users/zhaozhang/Downloads/AGORA_MW_analysis/AGORA_Data/z0/agora_z0_dataset_inspection.csv`


In [ ]:
rows = inspect_all(ROOT, OUTPUT_CSV)
inspection_df = pd.DataFrame(rows)
summary_cols = [
    "code_dir", "normalized_code", "load_ok", "current_redshift", "candidate_count",
    "N_gas_elem", "N_star", "N_DM_HR", "N_DM_total", "grid_count", "error",
]
display(inspection_df[[c for c in summary_cols if c in inspection_df.columns]])


## 2. Read the CSV, Display Component Counts/Masses, and Plot Mass Differences

This panel does not reload the simulations. It reads the saved CSV and displays the gas/grid element count, stellar particle count, high-resolution dark matter count, total dark matter count, and corresponding masses. The plot compares total component masses and mean element/particle masses between codes.


In [ ]:
df = pd.read_csv(OUTPUT_CSV)

numeric_cols = [
    "N_gas_elem", "M_gas_Msun", "gas_mean_mass_Msun",
    "N_star", "M_star_Msun", "star_mean_mass_Msun",
    "N_DM_HR", "M_DM_HR_Msun", "dm_hr_mean_mass_Msun",
    "N_DM_total", "M_DM_total_Msun", "dm_total_mean_mass_Msun",
    "grid_count", "domain_cell_count", "total_particles",
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

component_cols = [
    "code_dir", "normalized_code", "load_ok", "current_redshift",
    "N_gas_elem", "M_gas_Msun", "gas_mean_mass_Msun",
    "N_star", "M_star_Msun", "star_mean_mass_Msun",
    "N_DM_HR", "M_DM_HR_Msun", "dm_hr_mean_mass_Msun",
    "N_DM_total", "M_DM_total_Msun", "dm_total_mean_mass_Msun",
    "grid_count", "domain_cell_count", "component_notes",
]
component_table = df[[c for c in component_cols if c in df.columns]].copy()
display(component_table)

plot_df = df[df["load_ok"].astype(str).str.upper().isin(["TRUE", "True", "1"])].copy()
plot_df = plot_df.sort_values("code_dir")
labels = plot_df["code_dir"].to_numpy()
x = np.arange(len(labels))
width = 0.22

mass_panels = [
    ("M_gas_Msun", "Gas/grid mass"),
    ("M_star_Msun", "Stellar mass"),
    ("M_DM_HR_Msun", "High-resolution DM mass"),
    ("M_DM_total_Msun", "Total DM mass"),
]
mean_panels = [
    ("gas_mean_mass_Msun", "Mean gas/grid element mass"),
    ("star_mean_mass_Msun", "Mean stellar particle mass"),
    ("dm_hr_mean_mass_Msun", "Mean high-resolution DM particle mass"),
    ("dm_total_mean_mass_Msun", "Mean total DM particle mass"),
]

fig, axes = plt.subplots(2, 1, figsize=(12, 9), constrained_layout=True)

for i, (col, label) in enumerate(mass_panels):
    if col in plot_df.columns:
        axes[0].bar(x + (i - 1.5) * width, plot_df[col].to_numpy(dtype=float), width=width, label=label)
axes[0].set_yscale("log")
axes[0].set_ylabel(r"Total mass [$M_\odot$]")
axes[0].set_xticks(x, labels, rotation=30, ha="right")
axes[0].grid(True, axis="y", alpha=0.3)
axes[0].legend(ncol=2, frameon=False)
axes[0].set_title("AGORA z=0 component mass comparison")

for i, (col, label) in enumerate(mean_panels):
    if col in plot_df.columns:
        axes[1].bar(x + (i - 1.5) * width, plot_df[col].to_numpy(dtype=float), width=width, label=label)
axes[1].set_yscale("log")
axes[1].set_ylabel(r"Mean element/particle mass [$M_\odot$]")
axes[1].set_xticks(x, labels, rotation=30, ha="right")
axes[1].grid(True, axis="y", alpha=0.3)
axes[1].legend(ncol=2, frameon=False)

fig.savefig(OUTPUT_PNG, dpi=220)
plt.show()
print("Saved:", OUTPUT_PNG)
